# Chapter 41: Homographies

## Objective

In this tutorial we will build a homography estimator from scratch in PyTorch, validate it on synthetic point correspondences, study when the method works well or breaks down, and finish with a synthetic perspective-correction example.

By the end of the notebook you should be able to:

- explain what a planar homography does,
- move between Euclidean and homogeneous coordinates,
- estimate a homography with the normalized DLT algorithm,
- measure reprojection error,
- use RANSAC to reject outliers,
- warp a planar image patch with inverse warping,
- and recognize common failure modes such as degeneracy and overwhelming outlier rates.


## Plain-English Introduction

A **homography** is a 3x3 matrix that maps points from one image plane to another when the scene is planar, or when the camera motion can be approximated as a pure rotation around its optical center.

You can think of it as the algebra behind perspective effects such as:

- a rectangular poster appearing as a skewed quadrilateral in a photograph,
- a top-down floor plan being related to an oblique camera view,
- or one view of a flat checkerboard being warped into another.

Homographies matter because they let us:

- stitch panoramas,
- rectify documents and whiteboards,
- align planar surfaces across views,
- and initialize larger vision pipelines such as camera calibration and visual SLAM.

We will stay fully synthetic here so that the ground-truth transformation is known and every step is reproducible on CPU.


## The Math: Homogeneous Coordinates and $\tilde{\mathbf{p}}' \sim H\tilde{\mathbf{p}}$

A 2D point $\mathbf{p} = (x, y)$ becomes a **homogeneous** 3-vector by appending a 1:

$$
\tilde{\mathbf{p}} = \begin{bmatrix}x \\ y \\ 1\end{bmatrix}.
$$

A homography is a non-singular matrix

$$
H \in \mathbb{R}^{3 \times 3}, \qquad
\tilde{\mathbf{p}}' \sim H\tilde{\mathbf{p}}.
$$

The symbol $\sim$ means **equal up to scale**. In homogeneous coordinates, multiplying a point by any nonzero scalar does not change the represented Euclidean point. After applying $H$, we convert back to Euclidean coordinates by dividing by the last component:

$$
\tilde{\mathbf{p}}' = \begin{bmatrix}u \\ v \\ w\end{bmatrix}
\quad \Rightarrow \quad
\mathbf{p}' = \left(\frac{u}{w}, \frac{v}{w}\right).
$$

Because $H$ is defined only up to a scale factor, we should compare estimated and ground-truth homographies **after normalizing them to a common scale**.

To estimate $H$ from correspondences, we will use the **normalized Direct Linear Transform (DLT)**:

1. normalize source and destination points for numerical stability,
2. write the linear constraints implied by each correspondence,
3. solve the resulting homogeneous system with SVD,
4. denormalize the result back into the original coordinate system.


In [ ]:
import math
import random

import matplotlib.pyplot as plt
import torch


torch.set_default_dtype(torch.float64)
torch.manual_seed(7)
random.seed(7)

print("PyTorch version:", torch.__version__)
print("Default dtype:", torch.get_default_dtype())
print("Device: CPU")


## Synthetic Data Setup

We will create a regular 2D grid of points, define a known homography, and use it to generate perfect correspondences. This gives us a clean baseline before we introduce noise and outliers.


In [ ]:
def to_homogeneous(points: torch.Tensor) -> torch.Tensor:
    """Append a 1 to each 2D point."""
    if points.ndim != 2 or points.shape[1] != 2:
        raise ValueError("Expected points with shape (N, 2).")
    ones = torch.ones((points.shape[0], 1), dtype=points.dtype, device=points.device)
    return torch.cat([points, ones], dim=1)


def from_homogeneous(points: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    """Convert homogeneous points back to Euclidean coordinates."""
    if points.ndim != 2 or points.shape[1] != 3:
        raise ValueError("Expected homogeneous points with shape (N, 3).")
    scale = points[:, 2:].clone()
    scale = torch.where(scale.abs() < eps, torch.full_like(scale, eps), scale)
    return points[:, :2] / scale


def apply_homography(points: torch.Tensor, H: torch.Tensor) -> torch.Tensor:
    """Apply a 3x3 homography to 2D points."""
    if H.shape != (3, 3):
        raise ValueError("Expected H with shape (3, 3).")
    warped = to_homogeneous(points) @ H.T
    return from_homogeneous(warped)


def normalize_points(points: torch.Tensor, eps: float = 1e-12) -> tuple[torch.Tensor, torch.Tensor]:
    """Normalize points so the centroid is at the origin and mean distance is sqrt(2)."""
    centroid = points.mean(dim=0)
    centered = points - centroid
    mean_dist = torch.linalg.norm(centered, dim=1).mean().clamp(min=eps)
    scale = math.sqrt(2.0) / mean_dist

    T = torch.tensor(
        [
            [scale, 0.0, -scale * centroid[0]],
            [0.0, scale, -scale * centroid[1]],
            [0.0, 0.0, 1.0],
        ],
        dtype=points.dtype,
        device=points.device,
    )
    normalized = apply_homography(points, T)
    return normalized, T


def estimate_homography_dlt(src_points: torch.Tensor, dst_points: torch.Tensor) -> torch.Tensor:
    """Estimate a homography with normalized DLT."""
    if src_points.shape != dst_points.shape or src_points.shape[0] < 4:
        raise ValueError("Need at least four matching point pairs.")

    src_norm, T_src = normalize_points(src_points)
    dst_norm, T_dst = normalize_points(dst_points)

    rows = []
    for (x, y), (u, v) in zip(src_norm, dst_norm):
        rows.append(
            torch.tensor(
                [-x, -y, -1.0, 0.0, 0.0, 0.0, u * x, u * y, u],
                dtype=src_points.dtype,
                device=src_points.device,
            )
        )
        rows.append(
            torch.tensor(
                [0.0, 0.0, 0.0, -x, -y, -1.0, v * x, v * y, v],
                dtype=src_points.dtype,
                device=src_points.device,
            )
        )
    A = torch.stack(rows)

    _, _, vh = torch.linalg.svd(A)
    H_norm = vh[-1].reshape(3, 3)
    H = torch.linalg.inv(T_dst) @ H_norm @ T_src
    return H / torch.linalg.norm(H)


def compute_reprojection_error(H: torch.Tensor, src_points: torch.Tensor, dst_points: torch.Tensor) -> torch.Tensor:
    """Compute Euclidean reprojection error for each correspondence."""
    projected = apply_homography(src_points, H)
    return torch.linalg.norm(projected - dst_points, dim=1)


def ransac_homography(
    src_points: torch.Tensor,
    dst_points: torch.Tensor,
    threshold: float,
    num_iters: int,
) -> tuple[torch.Tensor, torch.Tensor, dict]:
    """Estimate a homography robustly with four-point RANSAC."""
    if src_points.shape[0] < 4:
        raise ValueError("RANSAC needs at least four correspondences.")

    n = src_points.shape[0]
    best_H = None
    best_inliers = None
    best_count = -1
    best_mean_error = float("inf")

    for _ in range(num_iters):
        sample_idx = torch.randperm(n)[:4]
        try:
            candidate_H = estimate_homography_dlt(src_points[sample_idx], dst_points[sample_idx])
        except RuntimeError:
            continue

        errors = compute_reprojection_error(candidate_H, src_points, dst_points)
        inliers = errors < threshold
        count = int(inliers.sum().item())
        mean_error = float(errors[inliers].mean().item()) if count > 0 else float("inf")

        if count > best_count or (count == best_count and mean_error < best_mean_error):
            best_H = candidate_H
            best_inliers = inliers
            best_count = count
            best_mean_error = mean_error

    if best_H is None or best_inliers is None or int(best_inliers.sum().item()) < 4:
        raise RuntimeError("RANSAC failed to find a valid homography.")

    refined_H = estimate_homography_dlt(src_points[best_inliers], dst_points[best_inliers])
    refined_errors = compute_reprojection_error(refined_H, src_points, dst_points)
    diagnostics = {
        "mean_inlier_error": float(refined_errors[best_inliers].mean().item()),
        "mean_all_error": float(refined_errors.mean().item()),
        "num_inliers": int(best_inliers.sum().item()),
        "inlier_ratio": float(best_inliers.double().mean().item()),
    }
    return refined_H, best_inliers, diagnostics


def normalize_homography_scale(H: torch.Tensor) -> torch.Tensor:
    """Normalize a homography so it can be compared up to scale."""
    if abs(float(H[-1, -1])) > 1e-12:
        return H / H[-1, -1]
    return H / torch.linalg.norm(H)


def make_grid(num_x: int = 6, num_y: int = 6, spacing: float = 1.0) -> torch.Tensor:
    xs = torch.linspace(-2.5, 2.5, steps=num_x) * spacing
    ys = torch.linspace(-2.0, 2.0, steps=num_y) * spacing
    yy, xx = torch.meshgrid(ys, xs, indexing="ij")
    return torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)


src_grid = make_grid()
H_gt = torch.tensor(
    [
        [1.10, 0.18, 1.20],
        [-0.12, 0.95, -0.60],
        [0.015, 0.020, 1.00],
    ]
)

dst_grid_clean = apply_homography(src_grid, H_gt)
print("Number of correspondences:", src_grid.shape[0])
print("Ground-truth H:")
print(normalize_homography_scale(H_gt))


## Visualizing the Known Transformation

The first plot shows the original 2D grid. The second plot shows how the same grid is warped by the known homography. This is the geometry we want our estimator to recover from noisy correspondences.


In [ ]:
def plot_grid(ax, points: torch.Tensor, title: str, color: str = "tab:blue") -> None:
    pts = points.reshape(6, 6, 2)
    for row in pts:
        ax.plot(row[:, 0], row[:, 1], color=color, alpha=0.75)
    for col in pts.permute(1, 0, 2):
        ax.plot(col[:, 0], col[:, 1], color=color, alpha=0.75)
    ax.scatter(points[:, 0], points[:, 1], s=24, color=color)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.grid(True, alpha=0.25)


fig, axes = plt.subplots(1, 2, figsize=(12, 5))
plot_grid(axes[0], src_grid, "Original 2D grid")
plot_grid(axes[1], dst_grid_clean, "Grid transformed by the ground-truth homography", color="tab:orange")
plt.tight_layout()
plt.show()


## Noisy Correspondences and Outliers

Real feature matches are never perfect. We will perturb the destination points with Gaussian noise and then replace a fraction of them with random outliers. This lets us test both least-squares estimation and RANSAC under controlled conditions.


In [ ]:
def generate_correspondences(
    H: torch.Tensor,
    noise_std: float = 0.05,
    outlier_ratio: float = 0.2,
    num_x: int = 6,
    num_y: int = 6,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    src = make_grid(num_x=num_x, num_y=num_y)
    dst_clean = apply_homography(src, H)
    dst_noisy = dst_clean + noise_std * torch.randn_like(dst_clean)

    num_points = src.shape[0]
    num_outliers = int(round(outlier_ratio * num_points))
    true_inliers = torch.ones(num_points, dtype=torch.bool)

    if num_outliers > 0:
        outlier_idx = torch.randperm(num_points)[:num_outliers]
        mins = dst_clean.min(dim=0).values - 1.0
        maxs = dst_clean.max(dim=0).values + 1.0
        random_points = mins + (maxs - mins) * torch.rand((num_outliers, 2), dtype=dst_noisy.dtype)
        dst_noisy[outlier_idx] = random_points
        true_inliers[outlier_idx] = False

    return src, dst_clean, dst_noisy, true_inliers


src_points, dst_clean, dst_points, true_inliers = generate_correspondences(
    H_gt,
    noise_std=0.08,
    outlier_ratio=0.25,
)

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(src_points[:, 0], src_points[:, 1], label="Source points", color="tab:blue", s=28)
ax.scatter(dst_points[:, 0], dst_points[:, 1], label="Noisy destination points", color="tab:red", s=28)
for i in range(src_points.shape[0]):
    ax.plot(
        [src_points[i, 0], dst_points[i, 0]],
        [src_points[i, 1], dst_points[i, 1]],
        color="gray",
        alpha=0.25,
        linewidth=1,
    )
ax.set_title("Synthetic noisy correspondences")
ax.set_aspect("equal")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

print("True inlier ratio:", true_inliers.double().mean().item())


## Estimation in PyTorch: DLT vs RANSAC

We will first estimate a homography from all correspondences using normalized DLT. Then we will run RANSAC to repeatedly sample four points, score the candidate by reprojection error, and refit on the best inlier set.


In [ ]:
H_dlt = estimate_homography_dlt(src_points, dst_points)
errors_dlt = compute_reprojection_error(H_dlt, src_points, dst_points)

H_ransac, predicted_inliers, ransac_info = ransac_homography(
    src_points,
    dst_points,
    threshold=0.20,
    num_iters=400,
)
errors_ransac = compute_reprojection_error(H_ransac, src_points, dst_points)

print("DLT mean reprojection error:   ", float(errors_dlt.mean().item()))
print("RANSAC mean reprojection error:", float(errors_ransac.mean().item()))
print("RANSAC diagnostics:", ransac_info)


## Visualizing RANSAC Inliers and Outliers

The plot below colors the matched destination points by whether RANSAC classified them as inliers or outliers. This gives a direct picture of why robust estimation improves the fit.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(dst_points[true_inliers, 0], dst_points[true_inliers, 1], color="tab:green", s=32, label="True inliers")
axes[0].scatter(dst_points[~true_inliers, 0], dst_points[~true_inliers, 1], color="tab:red", s=32, label="True outliers")
axes[0].set_title("Ground-truth inlier/outlier split")
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.25)
axes[0].legend()

axes[1].scatter(dst_points[predicted_inliers, 0], dst_points[predicted_inliers, 1], color="tab:green", s=32, label="RANSAC inliers")
axes[1].scatter(dst_points[~predicted_inliers, 0], dst_points[~predicted_inliers, 1], color="tab:red", s=32, label="RANSAC outliers")
axes[1].set_title("RANSAC classification")
axes[1].set_aspect("equal")
axes[1].grid(True, alpha=0.25)
axes[1].legend()

plt.tight_layout()
plt.show()


## Validation Against Ground Truth

A homography is only defined up to scale, so we normalize the estimated and true matrices before comparing them. We also report the mean reprojection error and the inlier ratio found by RANSAC.


In [ ]:
H_gt_scaled = normalize_homography_scale(H_gt)
H_dlt_scaled = normalize_homography_scale(H_dlt)
H_ransac_scaled = normalize_homography_scale(H_ransac)

matrix_error_dlt = torch.linalg.norm(H_gt_scaled - H_dlt_scaled).item()
matrix_error_ransac = torch.linalg.norm(H_gt_scaled - H_ransac_scaled).item()
mean_reproj_error = errors_ransac.mean().item()
inlier_ratio = predicted_inliers.double().mean().item()
precision = (predicted_inliers & true_inliers).sum().item() / max(predicted_inliers.sum().item(), 1)
recall = (predicted_inliers & true_inliers).sum().item() / max(true_inliers.sum().item(), 1)

print("Ground-truth H (normalized):")
print(H_gt_scaled)
print()
print("DLT estimate (normalized):")
print(H_dlt_scaled)
print()
print("RANSAC estimate (normalized):")
print(H_ransac_scaled)
print()
print("Frobenius error vs ground truth")
print("  DLT:   ", matrix_error_dlt)
print("  RANSAC:", matrix_error_ransac)
print()
print("Mean reprojection error:", mean_reproj_error)
print("RANSAC inlier ratio:   ", inlier_ratio)
print("RANSAC precision:      ", precision)
print("RANSAC recall:         ", recall)


## Perspective Correction with Inverse Warping

Estimating a homography is not only about matching sparse points. Once we know a mapping between two views of the same plane, we can also move **every pixel** on that plane. That makes homographies a natural tool for document rectification, whiteboard cleanup, and other perspective-correction tasks.

This educational homography demo uses a synthetic checkerboard so that the geometry stays easy to inspect. We will create a checkerboard image, warp it into a perspective view, and then undo that distortion with a second inverse-warping step.

A practical detail matters here: we prefer **inverse warping** over forward warping. Forward warping pushes source pixels into the destination image, which often leaves holes because destination pixels are skipped. Inverse warping does the opposite: for each destination pixel, it asks where that pixel came from in the source image, so every destination location gets a value whenever the mapped source coordinate stays inside the image.


In [ ]:
def create_checkerboard(
    height: int = 180,
    width: int = 180,
    num_checks_x: int = 9,
    num_checks_y: int = 9,
) -> torch.Tensor:
    """Create a grayscale checkerboard image with values in [0, 1]."""
    y = torch.arange(height, dtype=torch.get_default_dtype())
    x = torch.arange(width, dtype=torch.get_default_dtype())
    yy, xx = torch.meshgrid(y, x, indexing="ij")

    check_x = torch.clamp((xx * num_checks_x / width).floor().long(), max=num_checks_x - 1)
    check_y = torch.clamp((yy * num_checks_y / height).floor().long(), max=num_checks_y - 1)
    board = ((check_x + check_y) % 2).to(torch.get_default_dtype())
    return 0.15 + 0.8 * board


def bilinear_interpolate(image: torch.Tensor, x: torch.Tensor, y: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample an image at floating-point coordinates with bilinear interpolation."""
    if image.ndim == 2:
        image_3d = image.unsqueeze(-1)
        squeeze_channel = True
    elif image.ndim == 3:
        image_3d = image
        squeeze_channel = False
    else:
        raise ValueError("Expected image with shape (H, W) or (H, W, C).")

    height, width, _ = image_3d.shape
    x0 = torch.floor(x).long()
    y0 = torch.floor(y).long()
    x1 = x0 + 1
    y1 = y0 + 1

    valid = (x >= 0.0) & (x <= width - 1) & (y >= 0.0) & (y <= height - 1)

    x0c = x0.clamp(0, width - 1)
    x1c = x1.clamp(0, width - 1)
    y0c = y0.clamp(0, height - 1)
    y1c = y1.clamp(0, height - 1)

    Ia = image_3d[y0c, x0c]
    Ib = image_3d[y0c, x1c]
    Ic = image_3d[y1c, x0c]
    Id = image_3d[y1c, x1c]

    x0f = x0.to(image_3d.dtype)
    x1f = x1.to(image_3d.dtype)
    y0f = y0.to(image_3d.dtype)
    y1f = y1.to(image_3d.dtype)

    wa = (x1f - x) * (y1f - y)
    wb = (x - x0f) * (y1f - y)
    wc = (x1f - x) * (y - y0f)
    wd = (x - x0f) * (y - y0f)

    sampled = (
        wa.unsqueeze(-1) * Ia
        + wb.unsqueeze(-1) * Ib
        + wc.unsqueeze(-1) * Ic
        + wd.unsqueeze(-1) * Id
    )
    sampled = sampled * valid.unsqueeze(-1)

    if squeeze_channel:
        sampled = sampled.squeeze(-1)
    return sampled, valid


def inverse_warp_image(
    image: torch.Tensor,
    H_src_from_dst: torch.Tensor,
    out_height: int,
    out_width: int,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Warp an image by inverse mapping from destination pixels to source pixels."""
    y = torch.arange(out_height, dtype=image.dtype)
    x = torch.arange(out_width, dtype=image.dtype)
    yy, xx = torch.meshgrid(y, x, indexing="ij")

    dst_points = torch.stack([xx.reshape(-1), yy.reshape(-1)], dim=1)
    src_points = apply_homography(dst_points, H_src_from_dst)

    sampled, valid = bilinear_interpolate(image, src_points[:, 0], src_points[:, 1])
    if image.ndim == 2:
        warped = sampled.reshape(out_height, out_width)
    else:
        warped = sampled.reshape(out_height, out_width, image.shape[2])
    return warped, valid.reshape(out_height, out_width)


checkerboard = create_checkerboard()
board_height, board_width = checkerboard.shape

src_corners = torch.tensor(
    [
        [0.0, 0.0],
        [board_width - 1.0, 0.0],
        [board_width - 1.0, board_height - 1.0],
        [0.0, board_height - 1.0],
    ]
)

dst_corners = torch.tensor(
    [
        [28.0, 18.0],
        [198.0, 6.0],
        [176.0, 212.0],
        [12.0, 186.0],
    ]
)

H_checkerboard = estimate_homography_dlt(src_corners, dst_corners)
distorted_checkerboard, distorted_mask = inverse_warp_image(
    checkerboard,
    torch.linalg.inv(H_checkerboard),
    out_height=220,
    out_width=220,
)
rectified_checkerboard, rectified_mask = inverse_warp_image(
    distorted_checkerboard,
    H_checkerboard,
    out_height=board_height,
    out_width=board_width,
)

print("Checkerboard homography (normalized):")
print(normalize_homography_scale(H_checkerboard))
print("Distorted coverage ratio:", float(distorted_mask.double().mean().item()))
print("Rectified coverage ratio:", float(rectified_mask.double().mean().item()))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

axes[0].imshow(checkerboard, cmap="gray", vmin=0.0, vmax=1.0)
axes[0].set_title("Original checkerboard")
axes[0].axis("off")

axes[1].imshow(distorted_checkerboard, cmap="gray", vmin=0.0, vmax=1.0)
axes[1].scatter(dst_corners[:, 0], dst_corners[:, 1], color="tab:red", s=18)
axes[1].set_title("Perspective-distorted checkerboard")
axes[1].axis("off")

axes[2].imshow(rectified_checkerboard, cmap="gray", vmin=0.0, vmax=1.0)
axes[2].set_title("Rectified checkerboard")
axes[2].axis("off")

plt.tight_layout()
plt.show()

rectification_mae = (rectified_checkerboard[rectified_mask] - checkerboard[rectified_mask]).abs().mean().item()
print(f"Rectification mean absolute error: {rectification_mae:.4f}")


Bilinear interpolation is needed because the inverse-mapped source coordinates are almost never integers. A destination pixel often lands between four source pixels, so we blend those neighbors instead of rounding to the nearest one. That keeps the synthetic perspective-correction example visually smoother and avoids staircase artifacts.

This toy demo is still limited. The checkerboard is perfectly planar, the four corner correspondences are noise-free, there is no lens distortion, and the grayscale pattern has no occlusions or lighting changes. Real photographs add all of those complications, so this PyTorch implementation of DLT/RANSAC/inverse warping should be read as an educational homography demo rather than a full image stitching system.


## Parameter Study

Two questions matter in practice:

1. How does measurement noise affect reprojection error?
2. How does the outlier ratio affect RANSAC success?

The next experiments sweep over both parameters on synthetic data so we can see the trends directly.


In [ ]:
noise_levels = [0.0, 0.02, 0.05, 0.10, 0.20, 0.35]
noise_errors = []

for noise_std in noise_levels:
    trial_errors = []
    for _ in range(20):
        src, _, dst, _ = generate_correspondences(H_gt, noise_std=noise_std, outlier_ratio=0.0)
        H_est = estimate_homography_dlt(src, dst)
        trial_errors.append(compute_reprojection_error(H_est, src, dst).mean().item())
    noise_errors.append(sum(trial_errors) / len(trial_errors))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noise_levels, noise_errors, marker="o", linewidth=2)
ax.set_title("Noise level vs mean reprojection error")
ax.set_xlabel("Gaussian noise standard deviation")
ax.set_ylabel("Mean reprojection error")
ax.grid(True, alpha=0.3)
plt.show()

for noise_std, err in zip(noise_levels, noise_errors):
    print(f"noise_std={noise_std:>4.2f} -> mean error={err:.4f}")


In [ ]:
outlier_ratios = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
success_rates = []

for outlier_ratio in outlier_ratios:
    successes = 0
    for _ in range(20):
        src, _, dst, _ = generate_correspondences(H_gt, noise_std=0.08, outlier_ratio=outlier_ratio)
        try:
            H_est, inliers, _ = ransac_homography(src, dst, threshold=0.20, num_iters=500)
            H_est_scaled = normalize_homography_scale(H_est)
            H_gt_scaled = normalize_homography_scale(H_gt)
            matrix_error = torch.linalg.norm(H_est_scaled - H_gt_scaled).item()
            if matrix_error < 0.25:
                successes += 1
        except RuntimeError:
            pass
    success_rates.append(successes / 20)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(outlier_ratios, success_rates, marker="o", linewidth=2, color="tab:orange")
ax.set_title("Outlier ratio vs RANSAC success")
ax.set_xlabel("Outlier ratio")
ax.set_ylabel("Success rate")
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.3)
plt.show()

for outlier_ratio, rate in zip(outlier_ratios, success_rates):
    print(f"outlier_ratio={outlier_ratio:>4.2f} -> success rate={rate:.2f}")


## Failure Case: Degeneracy and Extreme Outliers

Homography estimation needs well-conditioned correspondences. Two common problems are:

- **Nearly collinear points**: the DLT system becomes poorly constrained.
- **Too many outliers**: RANSAC may not sample enough all-inlier minimal sets.

We will demonstrate both.


In [ ]:
collinear_src = torch.tensor(
    [
        [-2.0, -0.02],
        [-1.0,  0.00],
        [ 0.0,  0.01],
        [ 1.0,  0.02],
        [ 2.0,  0.03],
        [ 3.0,  0.05],
    ]
)
collinear_dst = apply_homography(collinear_src, H_gt)
collinear_dst = collinear_dst + 0.01 * torch.randn_like(collinear_dst)

H_collinear = estimate_homography_dlt(collinear_src, collinear_dst)
collinear_error = compute_reprojection_error(H_collinear, collinear_src, collinear_dst).mean().item()
collinear_matrix_error = torch.linalg.norm(
    normalize_homography_scale(H_collinear) - normalize_homography_scale(H_gt)
).item()

src_hard, _, dst_hard, _ = generate_correspondences(H_gt, noise_std=0.08, outlier_ratio=0.80)
try:
    H_hard, hard_inliers, hard_info = ransac_homography(src_hard, dst_hard, threshold=0.20, num_iters=500)
    hard_matrix_error = torch.linalg.norm(
        normalize_homography_scale(H_hard) - normalize_homography_scale(H_gt)
    ).item()
    hard_status = (
        f"succeeded, matrix error={hard_matrix_error:.3f}, "
        f"inlier ratio={hard_info['inlier_ratio']:.2f}"
    )
except RuntimeError as exc:
    hard_status = f"failed: {exc}"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(collinear_src[:, 0], collinear_src[:, 1], color="tab:blue", s=35)
axes[0].set_title("Nearly collinear source points")
axes[0].set_aspect("equal")
axes[0].grid(True, alpha=0.25)

axes[1].scatter(dst_hard[:, 0], dst_hard[:, 1], color="tab:red", s=28)
axes[1].set_title("Example with 80% outliers")
axes[1].set_aspect("equal")
axes[1].grid(True, alpha=0.25)

plt.tight_layout()
plt.show()

print("Nearly collinear case")
print("  Mean reprojection error:", collinear_error)
print("  Matrix error vs ground truth:", collinear_matrix_error)
print()
print("Extreme-outlier RANSAC case")
print(" ", hard_status)


## Summary and Reproducibility Notes

We used PyTorch to implement the full homography estimation pipeline from scratch:

- homogeneous coordinate conversion,
- point normalization,
- normalized DLT,
- reprojection error,
- RANSAC-based robust fitting,
- and inverse image warping for a synthetic perspective-correction example.

The synthetic experiments highlighted four useful takeaways:

1. normalized DLT works well when correspondences are clean and well spread out,
2. RANSAC is essential when mismatches are present,
3. inverse warping turns the estimated homography into a dense image-rectification tool,
4. degeneracy and extreme outlier rates can still defeat the estimator.

### Reproducibility

- The notebook uses only synthetic data.
- Random seeds are fixed at the top of the notebook.
- All computations are CPU-friendly.
- The implementation avoids OpenCV for core estimation and core inverse warping so the math is easy to inspect.

A natural extension is to replace the synthetic corners with real detections and study how interpolation, occlusions, and imperfect corner localization affect rectification quality.
